<a href="https://colab.research.google.com/github/Ravinduranasinghe/yt-dlp-colab/blob/main/yt_dlp_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, subprocess, re
from google.colab import drive, files

# ============================================================
#  CELL 1 — Setup
# ============================================================
def setup():
    print("🔧 Installing Node.js 20...")
    subprocess.run('curl -fsSL https://deb.nodesource.com/setup_20.x | bash -', shell=True, capture_output=True)
    subprocess.run(['apt-get', 'install', '-y', 'nodejs'], capture_output=True)

    node = subprocess.run(['node', '--version'], capture_output=True, text=True)
    print(f"✅ node {node.stdout.strip()}")

    print("🔧 Installing yt-dlp + yt-dlp-ejs solver scripts...")
    subprocess.run(
        ['pip', 'install', '-U', '--pre', 'yt-dlp[default]', '--break-system-packages'],
        check=True
    )

    yt = subprocess.run(['yt-dlp', '--version'], capture_output=True, text=True)
    print(f"✅ yt-dlp {yt.stdout.strip()}")
    print("✅ Setup complete.")

setup()

# ============================================================
#  CONFIGURATION
# ============================================================
playlist_url = "https://www.youtube.com/playlist?list=PLKr-lYdiK6lLGtymt5IWqxWOOXdrbs6SA"

# ============================================================
#  STEP 1 — Upload cookies.txt
# ============================================================
print("\n📂 Upload your 'cookies.txt' file.")
print("   ⚠️  Use cookies from the account that OWNS or has access to this playlist.\n")
uploaded = files.upload()
if not uploaded:
    raise Exception("No cookie file uploaded.")
cookie_file = list(uploaded.keys())[0]
print(f"✅ Cookie file: {cookie_file}")

# ============================================================
#  STEP 2 — Mount Google Drive
# ============================================================
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print("✅ Drive already mounted.")

# ============================================================
#  STEP 3 — Fetch playlist title & create save folder
# ============================================================
print("\n🔍 Fetching playlist info...")
raw_title_cmd = subprocess.run(
    ['yt-dlp',
     '--cookies', cookie_file,
     '--flat-playlist',
     '--get-filename', '-o', '%(playlist_title)s',
     '--playlist-items', '1',
     playlist_url],
    capture_output=True, text=True
)
raw_title  = raw_title_cmd.stdout.strip() or "Playlist"
clean_name = re.sub(r'[\\/*?:"<>|]', "_", raw_title)
save_path  = f"/content/drive/MyDrive/{clean_name}"
os.makedirs(save_path, exist_ok=True)
print(f"✅ Playlist: '{raw_title}'")
print(f"   Saving to: Drive/{clean_name}")

archive_file = f"{save_path}/.download_archive.txt"

if os.path.exists(archive_file):
    with open(archive_file) as f:
        skip_count = sum(1 for line in f if line.strip())
    print(f"\n📋 Archive: {skip_count} already downloaded (will skip)")
else:
    print(f"\n📋 No archive found — starting fresh.")

# ============================================================
#  STEP 4 — Count playlist entries before downloading
# ============================================================
print("\n🔢 Counting playlist entries (paginating full playlist)...")
count_cmd = subprocess.run(
    ['yt-dlp',
     '--cookies', cookie_file,
     '--flat-playlist',
     '--no-lazy-playlist',
     '--extractor-retries', 'infinite',
     '--print', 'id',
     playlist_url],
    capture_output=True, text=True
)
all_ids = [line.strip() for line in count_cmd.stdout.splitlines() if line.strip()]
print(f"✅ {len(all_ids)} video(s) visible in playlist")

# ============================================================
#  STEP 5 — Download + transcode to 48k opus
# ============================================================
print(f"\n⬇️  Downloading…\n")

empty_trash_cmd = 'rm -rf ~/.local/share/Trash/files/* ~/.local/share/Trash/info/* 2>/dev/null; true'

cmd = [
    'yt-dlp',
    '--cookies',        cookie_file,
    '--extractor-args', 'youtube:player_client=web',
    '--js-runtimes',    'node',

    '--no-lazy-playlist',
    '--yes-playlist',
    '--extractor-retries', 'infinite',

    '--download-archive', archive_file,

    # CHANGED: prioritize native Opus stream to avoid downloading large mp4/m4a
    '-f',                  'bestaudio[acodec=opus]/bestaudio[ext=webm]/bestaudio/best',
    '-x',
    '--audio-format',      'opus',
    '--audio-quality',     '10',
    '--postprocessor-args', (
        'FFmpegExtractAudio:'
        '-c:a libopus '
        '-b:a 48k '
        '-vbr on '
        '-compression_level 10 '
        '-application audio'
    ),

    '--add-metadata',
    '--no-embed-thumbnail',
    '--windows-filenames',
    '--no-overwrites',
    '--no-abort-on-error',
    '--sleep-interval',     '5',
    '--max-sleep-interval', '12',

    '--exec', f'after_move:{empty_trash_cmd}',

    '-o', f'{save_path}/%(playlist_index)s - %(title)s.%(ext)s',

    playlist_url
]

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

# Final trash empty in case anything was left over
subprocess.run(empty_trash_cmd, shell=True)

# ============================================================
#  CLEANUP
# ============================================================
if os.path.exists(cookie_file):
    os.remove(cookie_file)
    print("\n🗑️  Cookie file deleted.")

if os.path.exists(archive_file):
    with open(archive_file) as f:
        total = sum(1 for line in f if line.strip())
    print(f"📋 Archive updated: {total} video(s) in skip list.")

print(f"\n✅ Done! Files saved to: Drive/{clean_name}")
print(f"📁 Archive: Drive/{clean_name}/.download_archive.txt")
print()
print("─" * 60)
print("💡 ARCHIVE REMINDER")
print("─" * 60)
print("The archive tracks completed downloads so you can cancel")
print("and resume without re-downloading. Do NOT delete it.")
print()
print("To re-download a video you previously deleted from Drive:")
print("  1. Open the archive file in Drive")
print("  2. Find the line:  youtube VIDEOID")
print("     (VIDEOID = the ?v= part of the YouTube URL)")
print("  3. Delete that line and save")
print("  4. Run this script again")